In [102]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
df = pd.DataFrame({"col1":[-0.73,-2.36, 2.17, np.nan, -0.82, "1.83",  1.66, np.nan,  0.78 , 1.56, 2.82, 3.55],
                   "col2":[ 0.21, -0.089 , -1.34,  2.57,  1.81, -1.77, -0.28, -0.73, -0.99, -1.54, 0.45, -0.67],
                   "col3":[-0.92, np.nan,  1.92 , -1.42 , -0.38, 1.47,  np.nan, -1.79,  np.nan,  1.22, 0.56, -0.34],
                   "cat":["B", "B", "M", "B", "B", "M", np.nan, "B", "M", "M", "B", "M"],
                   "peso":[80, 70, 50, 60, 90, 55, 65, 85, 75, 45, 68, 72],
                   "altura":[1.80, 1.75, 1.60, 1.70, 1.85, 1.65, 1.68, 1.82, 1.78, 1.55, 1.72, 1.74],
                   "target":[0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1]})


Función para ajustar los tipos erroneos.

In [103]:
def type_normalizer(X):
    X = X.copy()
    X["col1"] = X["col1"].astype(float)
    return X    

def calcula_IMC(X):
    X = X.copy()
    return pd.DataFrame({"IMC": X["peso"] / (X["altura"] ** 2)})



In [104]:
#sns.pairplot(df[["col1", "col2", "col3", "target"]], hue='target')
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3, test_size=0.3)

Function Transformer: genera una transformación a partir de una "llamable", normalmente una función u objeto de clase que implemente "\_\_call\_\_"


In [105]:
plcol1 = Pipeline(steps=[
    ("dtype_converter", FunctionTransformer(type_normalizer)),
    ("mean_col1", SimpleImputer(strategy="mean"))
])

In [106]:
ct = ColumnTransformer([("mean_col1", plcol1, ["col1"]),
                        ("median_col2", SimpleImputer(strategy="median") , ["col2"]),
                        ("most_freq_cat", SimpleImputer(strategy="most_frequent") , ["cat"]),
                        ("IMC", FunctionTransformer(calcula_IMC, validate=False), ["peso", "altura"])
                        ])

In [107]:
ct

ColumnTransformer(transformers=[('mean_col1',
                                 Pipeline(steps=[('dtype_converter',
                                                  FunctionTransformer(func=<function type_normalizer at 0x000001BA12277520>)),
                                                 ('mean_col1',
                                                  SimpleImputer())]),
                                 ['col1']),
                                ('median_col2',
                                 SimpleImputer(strategy='median'), ['col2']),
                                ('most_freq_cat',
                                 SimpleImputer(strategy='most_frequent'),
                                 ['cat']),
                                ('IMC',
                                 FunctionTransformer(func=<function calcula_IMC at 0x000001BA12E7C3A0>),
                                 ['peso', 'altura'])])

In [108]:
ct.fit_transform(X_train)

array([[3.55, -0.67, 'M', 23.781212841854934],
       [1.66, -0.28, 'B', 23.030045351473927],
       [1.6066666666666667, -0.73, 'B', 25.661152034778407],
       [-0.73, 0.21, 'B', 24.691358024691358],
       [1.6066666666666667, 2.57, 'B', 20.761245674740486],
       [1.56, -1.54, 'M', 18.730489073881373],
       [0.78, -0.99, 'M', 23.671253629592222],
       [2.82, 0.45, 'B', 22.985397512168742]], dtype=object)

In [109]:
pl = Pipeline(steps=[("ct", ct),
                     ("est", KNeighborsClassifier())])


In [110]:
pl

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('mean_col1',
                                                  Pipeline(steps=[('dtype_converter',
                                                                   FunctionTransformer(func=<function type_normalizer at 0x000001BA12277520>)),
                                                                  ('mean_col1',
                                                                   SimpleImputer())]),
                                                  ['col1']),
                                                 ('median_col2',
                                                  SimpleImputer(strategy='median'),
                                                  ['col2']),
                                                 ('most_freq_cat',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['cat']),
                                                 ('IMC',
                                                  FunctionTransformer(func=<function calcula_IMC at 0x000001BA12E7C3A0>),
                                                  ['peso', 'altura'])])),
                ('est', KNeighborsClassifier())])

In [111]:
pl.fit(X_train, y_train)
pl.predict(X_test)

ValueError: could not convert string to float: 'M'

Me faltaría convertir la categórica a numérica....pero perdí el nombre de la columna, puedo usar el número de la columna pero mejor aun, meter otro pipeline dentro del transformer

In [120]:
temp = pl[:-1].fit_transform(X_train)
temp

array([[3.55, -0.67, 'M', 23.781212841854934],
       [1.66, -0.28, 'B', 23.030045351473927],
       [1.6066666666666667, -0.73, 'B', 25.661152034778407],
       [-0.73, 0.21, 'B', 24.691358024691358],
       [1.6066666666666667, 2.57, 'B', 20.761245674740486],
       [1.56, -1.54, 'M', 18.730489073881373],
       [0.78, -0.99, 'M', 23.671253629592222],
       [2.82, 0.45, 'B', 22.985397512168742]], dtype=object)

In [126]:
plcat = Pipeline(steps=[
    ("most_freq_cat", SimpleImputer(strategy="most_frequent")),
    ("OneHot", OneHotEncoder(sparse_output=False, dtype=int, drop='first'))
])

ct = ColumnTransformer([("mean_col1", plcol1, ["col1"]),
                        ("median_col2", SimpleImputer(strategy="median") , ["col2"]),
                        ("cat", plcat , ["cat"]),
                        ("IMC", FunctionTransformer(calcula_IMC, validate=False), ["peso", "altura"])
                        ])

In [127]:
temp = ct.fit_transform(X_train)
temp

array([[ 3.55      , -0.67      ,  1.        , 23.78121284],
       [ 1.66      , -0.28      ,  0.        , 23.03004535],
       [ 1.60666667, -0.73      ,  0.        , 25.66115203],
       [-0.73      ,  0.21      ,  0.        , 24.69135802],
       [ 1.60666667,  2.57      ,  0.        , 20.76124567],
       [ 1.56      , -1.54      ,  1.        , 18.73048907],
       [ 0.78      , -0.99      ,  1.        , 23.67125363],
       [ 2.82      ,  0.45      ,  0.        , 22.98539751]])

In [128]:
pl = Pipeline(steps=[("ct", ct),
                     ("est", KNeighborsClassifier())])

temp = pl[:-1].fit_transform(X_train)
temp

array([[ 3.55      , -0.67      ,  1.        , 23.78121284],
       [ 1.66      , -0.28      ,  0.        , 23.03004535],
       [ 1.60666667, -0.73      ,  0.        , 25.66115203],
       [-0.73      ,  0.21      ,  0.        , 24.69135802],
       [ 1.60666667,  2.57      ,  0.        , 20.76124567],
       [ 1.56      , -1.54      ,  1.        , 18.73048907],
       [ 0.78      , -0.99      ,  1.        , 23.67125363],
       [ 2.82      ,  0.45      ,  0.        , 22.98539751]])

In [129]:
column_types = [type(temp[0, i]) for i in range(temp.shape[1])]
print(column_types)  # [<class 'int'>, <class 'str'>, <class 'float'>]

[<class 'numpy.float64'>, <class 'numpy.float64'>, <class 'numpy.float64'>, <class 'numpy.float64'>]


In [130]:
pl

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('mean_col1',
                                                  Pipeline(steps=[('dtype_converter',
                                                                   FunctionTransformer(func=<function type_normalizer at 0x000001BA12277520>)),
                                                                  ('mean_col1',
                                                                   SimpleImputer())]),
                                                  ['col1']),
                                                 ('median_col2',
                                                  SimpleImputer(strategy='median'),
                                                  ['col2']),
                                                 ('cat',
                                                  Pipeline(steps=[('most_freq_cat',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('OneHot',
                                                                   OneHotEncoder(drop='first',
                                                                                 dtype=<class 'int'>,
                                                                                 sparse_output=False))]),
                                                  ['cat']),
                                                 ('IMC',
                                                  FunctionTransformer(func=<function calcula_IMC at 0x000001BA12E7C3A0>),
                                                  ['peso', 'altura'])])),
                ('est', KNeighborsClassifier())])

In [131]:
pl.fit(X_train, y_train)
y_pred = pl.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



¿Faltó alguna transformación?